# Thesis figures - all charts & qualitative images (baseline: run8)

## 1. Install & mount

In [ ]:
!nvidia-smi
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')


## 2. Config

In [ ]:
IMGSZ = 1280
RUN   = 'run8'                 # baseline = best run (AdamW + cosine)
DRIVE = '/content/drive/MyDrive'
RUNS  = f'{DRIVE}/run8'        # wrapper folder that now holds run8, run8_ft, run8_demo, ...
DS    = '/content/iadd_subset_v5'
DATA  = f'{DS}/data.yaml'

# resolve the run8 wrapper to its real path (works whether it is a real MyDrive folder or a shortcut)
import glob as _g, os as _o
RUNS = next((c for c in [f'{DRIVE}/run8'] + _g.glob('/content/drive/.shortcut-targets-by-id/*/run8')
             if _o.path.isdir(c)), f'{DRIVE}/run8')
print('RUNS ->', RUNS)
print('best.pt exists:', _o.path.exists(f'{RUNS}/{RUN}/weights/best.pt'))   # sanity check


## 3. Unzip dataset (needed for the demo images)

In [ ]:
!rm -rf /content/iadd_subset_v5
!unzip -q "/content/drive/MyDrive/iadd_subset_v5.zip" -d /content


## 4. Per-class metrics - table + F1 & mAP bar charts (supervisor asked for the F1 table)

In [ ]:
from ultralytics import YOLO
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

m = YOLO(f'{RUNS}/{RUN}/weights/best.pt')
r = m.val(data=DATA, split='test', verbose=False)
ci = r.box.ap_class_index
rows = [(m.names[c], r.box.p[i], r.box.r[i], r.box.f1[i], r.box.ap50[i], r.box.ap[i])
        for i, c in enumerate(ci)]

print(f'{"class":14s}{"P":>7}{"R":>7}{"F1":>7}{"mAP50":>8}{"mAP50-95":>10}')
for n, p, rc, f1, a50, a in rows:
    print(f'{n:14s}{p:7.3f}{rc:7.3f}{f1:7.3f}{a50:8.3f}{a:10.3f}')

names = [x[0] for x in rows]
def bar(vals, title, fname, color):
    plt.figure(figsize=(8, 4)); plt.bar(names, vals, color=color)
    plt.ylim(0, 1); plt.title(title); plt.xticks(rotation=20)
    for i, v in enumerate(vals): plt.text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)
    plt.tight_layout(); plt.savefig(f'{RUNS}/{fname}', dpi=150); plt.close()
bar([x[3] for x in rows], 'F1 per class (test)',      'fig_perclass_F1.png',  '#2e75b6')
bar([x[4] for x in rows], 'mAP@0.5 per class (test)', 'fig_perclass_mAP.png', '#2e7d32')
print('saved fig_perclass_F1.png , fig_perclass_mAP.png ->', RUNS)


## 5. Run comparison chart (val vs test across runs)

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import numpy as np

results = {                         # (val, test) mAP@0.5 - fill as each run finishes
    'run7 (SGD)':       (0.846, 0.867),
    'run8 (AdamW+cos)': (0.849, 0.874),
    # 'run9 (mixup off)': (v, t),
    # 'run10 (medium)':   (v, t),
}
runs = list(results); val = [results[k][0] for k in runs]; test = [results[k][1] for k in runs]
x = np.arange(len(runs)); w = 0.38
plt.figure(figsize=(7, 4.5))
plt.bar(x - w/2, val, w, label='val', color='#2e75b6')
plt.bar(x + w/2, test, w, label='test', color='#e08a00')
plt.xticks(x, runs, rotation=10); plt.ylim(0.80, 0.90); plt.ylabel('mAP@0.5'); plt.legend()
for i in range(len(runs)):
    plt.text(x[i]-w/2, val[i]+0.002, f'{val[i]:.3f}', ha='center', fontsize=8)
    plt.text(x[i]+w/2, test[i]+0.002, f'{test[i]:.3f}', ha='center', fontsize=8)
plt.tight_layout(); plt.savefig(f'{RUNS}/fig_run_comparison.png', dpi=150); plt.close()
print('saved fig_run_comparison.png ->', RUNS)


## 6. Qualitative demo - diverse conditions + rare classes (not car-heavy)

In [ ]:
import glob, os, random
from collections import defaultdict
from ultralytics import YOLO
from PIL import Image
random.seed(0)
RARE = {2, 3, 5}                    # motorcycle, bus, traffic_light (the hard/interesting ones)
model = YOLO(f'{RUNS}/{RUN}/weights/best.pt')
OUT = f'{RUNS}/{RUN}_figs'; os.makedirs(OUT, exist_ok=True)

by_cond = defaultdict(list)
for p in glob.glob(f'{DS}/images/test/*.jpg'):
    cond = os.path.basename(p).split('__')[0].split('_')[-1]     # D / N / R / A
    by_cond[cond].append(p)

picks = []
for cond, ps in sorted(by_cond.items()):
    scored = []
    for p in random.sample(ps, min(80, len(ps))):
        r = model.predict(p, imgsz=IMGSZ, conf=0.35, verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0: continue
        cls = set(int(c) for c in r.boxes.cls.cpu().numpy())
        cf = r.boxes.conf.cpu().numpy()
        if cf.mean() < 0.6 or not (2 <= len(r.boxes) <= 15): continue
        score = len(cls) + 3*len(cls & RARE)        # reward class variety + rare classes
        scored.append((score, p, r))
    scored.sort(key=lambda t: t[0], reverse=True)
    picks += [(cond,) + t[1:] for t in scored[:4]]  # 4 per condition (day/night/rain/other)

print('picked per condition:', {c: sum(1 for x in picks if x[0] == c) for c in by_cond})
saved = []
for cond, p, r in picks:
    out = f'{OUT}/{cond}_{os.path.basename(p)}'; r.save(filename=out); saved.append(out)

cols = 4; rows = (len(saved) + cols - 1)//cols; w, h = 480, 270
sheet = Image.new('RGB', (w*cols, h*rows), 'white')
for k, sp in enumerate(saved):
    sheet.paste(Image.open(sp).convert('RGB').resize((w, h)), ((k % cols)*w, (k // cols)*h))
sheet.save(f'{OUT}/_contact_sheet.jpg', quality=92)
print('saved', len(saved), 'diverse demo images + _contact_sheet.jpg ->', OUT)
